# MapLibreum: Real-time Data and Events

This notebook demonstrates how to load live data sources that automatically update, implement camera animations, and handle map events like clicks and hovers.

## 1. Real-time Data Sources

MapLibreum supports `RealTimeDataSource`, which periodically polls a URL to fetch new GeoJSON data. This is useful for vehicle tracking or live telemetry.

In [ ]:
from maplibreum import Map
from maplibreum.realtime import RealTimeDataSource

m1 = Map(center=[-77.0364, 38.8951], zoom=13) # Washington D.C.

# In a real app, this URL would return fresh coordinates every second
live_data_url = "https://wanderdrone.appspot.com/" # A public test endpoint returning moving drone coordinates

# Create a live GeoJSON source that updates every 2 seconds (2000 ms)
live_source = RealTimeDataSource(
    data=live_data_url,
    interval=2000
)

# Add the source to the map
m1.add_source("drone_source", live_source)

# You still need to add a Layer to visualize the source data
m1.add_layer({
    "id": "drone_layer",
    "type": "symbol",
    "source": "drone_source",
    "layout": {
        "icon-image": "rocket-15", # A standard icon from MapLibre's default sprites
        "icon-size": 1.5
    }
})

m1

## 2. Animated Camera Movement (Fly To)

Instead of just snapping to a new location, you can animate the camera with the `fly_to` feature when real-time data arrives. We can demonstrate this via the specialized `RandomCoordinateFetcher` built for testing.

In [ ]:
from maplibreum.realtime import RandomCoordinateFetcher

m2 = Map(center=[0, 0], zoom=4)

# The RandomCoordinateFetcher automatically pulls random points 
# and we set it to fly the camera to each new point.
random_tracker = RandomCoordinateFetcher(
    source_id="random_points",
    interval=3000, # Move every 3 seconds
    fly_to=True    # Animate camera to the new coordinates
)

m2.add_source("random_points", random_tracker)

m2.add_layer({
    "id": "random_point_layer",
    "type": "circle",
    "source": "random_points",
    "paint": {
        "circle-color": "#ff0000",
        "circle-radius": 10,
        "circle-stroke-width": 2,
        "circle-stroke-color": "#ffffff"
    }
})

m2

## 3. Handling Map Events

You can interact with map features using `Popup` or `Tooltip`. In this example we'll make a popup automatically appear when you click on specific layers.

In [ ]:
import urllib.request
import json
from maplibreum import Map, GeoJson, GeoJsonPopup

# Load earthquake data from USGS
quake_url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
with urllib.request.urlopen(quake_url) as response:
    quake_data = json.loads(response.read().decode())

m3 = Map(center=[-119.4179, 36.7783], zoom=5) # California

# Create a popup that displays magnitude and place
quake_popup = GeoJsonPopup(
    fields=['mag', 'place'],
    aliases=['Magnitude:', 'Location:'],
    labels=True
)

# Define simple styling (larger circles for higher magnitude)
def style_quake(feature):
    mag = feature['properties'].get('mag', 0)
    if mag > 4:
        color = "#ff0000"
    elif mag > 2:
        color = "#ffaa00"
    else:
        color = "#00aa00"
        
    return {
        'color': color,
        'weight': 1,
        'fillColor': color,
        'fillOpacity': 0.6,
        'radius': max(3, mag * 2) # Base radius on magnitude
    }

# Note: To render points as circles instead of default markers in GeoJson,
# MapLibreum allows using the `pointToLayer` configuration internally via GeoJson.
GeoJson(
    data=quake_data,
    name="Earthquakes",
    popup=quake_popup,
    style_function=style_quake
).add_to(m3)

m3